In this work, we use transformer model to integrate gene expression and TCR amino acid sequences

Getting gene data

In [1]:
# %matplotlib inline

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np

import pandas as pd
# import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc

import anndata as ad

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

sc.settings.verbosity = 3


In [2]:
gene_TCR = ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides.h5ad')
gene_TCR

c:\Users\phill\anaconda3\envs\tensorflow\lib\site-packages\anndata\_core\anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [3]:
gene_TCR = gene_TCR[gene_TCR.obs.donor == 'donor1']
gene_TCR

View of AnnData object with n_obs × n_vars = 35515 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', '

In [4]:
gene_TCR.obs.donor

barcode
AGGGTGAGTATTACCG-18    donor1
CTTGGCTTCGTTGCCT-25    donor1
ACGATACTCGCAGGCT-40    donor1
ACGCCAGTCATGTCTT-8     donor1
TTCTTAGCAAAGAATC-4     donor1
                        ...  
GTGAAGGGTACCGTAT-36    donor1
CTTACCGCAATCAGAA-15    donor1
CGACTTCGTAAGTTCC-24    donor1
GGACAGATCGTGTAGT-7     donor1
TGTTCCGGTTCAGGCC-6     donor1
Name: donor, Length: 35515, dtype: category
Categories (1, object): ['donor1']

In [5]:
gene = pd.DataFrame(gene_TCR.X.todense())
gene

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.693147,0.0,0.0,0.0,0.0,0.000000,0.0,0.693147,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,1.791759,0.693147,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.693147,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.693147,0.000000,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,1.791759,0.000000,0.0,0.0
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.693147,0.0,0.0,0.000000,1.609438,0.000000,0.0,0.0
4,0.0,1.386294,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,1.386294,0.693147,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35510,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.693147,0.0,0.000000,0.0,0.0,0.000000,0.693147,1.386294,0.0,0.0
35511,0.0,0.693147,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,1.609438,1.945910,0.0,0.0
35512,0.0,0.000000,0.0,0.0,0.0,0.0,0.693147,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,1.609438,0.000000,1.098612,0.0,0.0
35513,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,1.609438,1.098612,0.0,0.0


In [6]:
tcr_seq = gene_TCR.obs[['cdr3_TRB']]
tcr_seq

,cdr3_TRB
barcode,
AGGGTGAGTATTACCG-18,CSAPSGEGRDTQYF
CTTGGCTTCGTTGCCT-25,CASSLFDSQETQYF
ACGATACTCGCAGGCT-40,CASSLFDSGRLDTQYF
ACGCCAGTCATGTCTT-8,CSASPGDYEQYF
TTCTTAGCAAAGAATC-4,CASSHGKGGNEQFF
...,...
GTGAAGGGTACCGTAT-36,CSASPGDYEQYF
CTTACCGCAATCAGAA-15,CASSIGLAGARELFF
CGACTTCGTAAGTTCC-24,CASAGNTEAFF


In [7]:
import tensorflow as tf

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
from tensorflow.keras.initializers import HeNormal
# Define input layer
input_gex = Input(shape=(100,))
gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
gex = Reshape(target_shape=(8,8,1))(gex)

# Convolutional layers
gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

gex = Flatten()(gex)
hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# Transposed Convolutional layers
tcr = Reshape(target_shape=(15,15,1))(hidden_layer)
tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Dropout(rate=0.2)(tcr)
tcr = Flatten()(tcr)
tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
tcr = Dense(units=121, activation='relu')(tcr)
# Define model
model = Model(inputs=input_gex, outputs=tcr)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
              loss='mse')

# Check layer names
model.summary()

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
# from tensorflow.keras.initializers import HeNormal

# # Define input layer
# input_gex = Input(shape=(100,))
# gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
# gex = Reshape(target_shape=(8,8,1))(gex)

# # Convolutional layers
# gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
# gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

# gex = Flatten()(gex)
# hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# # Transposed Convolutional layers
# tcr = Reshape(target_shape=(15,15,1))(hidden_layer)
# tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
# tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
# tcr = Dropout(rate=0.2)(tcr)
# tcr = Flatten()(tcr)
# tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
# tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
# tcr = Dense(units=121, activation='relu')(tcr)
# # Define model
# model = Model(inputs=input_gex, outputs=tcr)
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
#               loss='mse')

# # Check layer names
# model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 dense (Dense)               (None, 64)                6464      
                                                                 
 reshape (Reshape)           (None, 8, 8, 1)           0         
                                                                 
 conv2d (Conv2D)             (None, 8, 8, 64)          640       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 32)          18464     
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dense_1 (Dense)             (None, 225)               461025

In [9]:
AE_tcr = pd.read_csv("../AE_emb_TRB_all_peptides_10X_donor_1_only.csv")
AE_tcr

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,-1541.692100,431.963380,-113.487510,363.839600,-617.46550,-576.387700,-152.86124,-985.72640,445.512240,-236.100420,...,28.279720,1018.37134,261.22320,87.342865,-228.999980,146.428400,532.794070,865.979250,-578.558960,855.61690
1,-273.513100,-398.893860,616.804300,-877.823500,-871.22670,375.483060,422.57904,192.50789,-199.132490,-265.096560,...,494.893800,258.40955,-404.77722,-428.136350,-191.204730,-372.523960,203.390100,1186.992800,96.218480,-457.06020
2,380.871000,58.110577,-132.314600,-259.367740,-274.57843,-194.467930,248.31584,265.90674,-60.173637,-98.568726,...,92.126270,29.88193,-891.70480,370.281830,59.372814,-729.678000,902.504940,1161.101400,-700.613340,-425.83978
3,-1229.993200,261.055080,-116.357920,339.648250,-440.47760,65.392690,-221.38164,-939.52580,145.774370,39.421880,...,153.995830,770.41030,-417.14660,194.590100,-287.056100,-4.876206,-74.260704,43.650208,-67.906160,-309.77994
4,-1113.337300,-93.408485,-280.131400,-150.605300,-93.47611,-520.023130,64.99572,-1008.05770,174.263440,-62.470676,...,-56.364480,984.19050,467.82294,247.904310,409.301880,-60.920048,744.377300,1218.065600,-456.735320,617.48610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35510,-1229.993200,261.055080,-116.357920,339.648250,-440.47760,65.392690,-221.38164,-939.52580,145.774370,39.421880,...,153.995830,770.41030,-417.14660,194.590100,-287.056100,-4.876206,-74.260704,43.650208,-67.906160,-309.77994
35511,-24.937555,-610.643400,231.669100,-48.156483,242.74973,-65.198640,-660.69820,-419.44427,-597.105000,92.525670,...,-531.762940,931.85034,-257.47950,351.656740,-311.497500,-156.879900,-360.937260,744.809800,74.583550,567.55290
35512,328.290000,477.070500,35.919773,-269.693700,308.35275,-121.531600,-569.52875,-509.88388,-480.568760,210.404070,...,509.689030,871.91560,447.09885,-565.088870,-756.617900,-695.267200,170.006210,966.821960,51.627117,-1258.25700
35513,66.083610,-979.693660,-210.645480,-364.229980,396.45697,-290.372100,330.68960,-123.65221,-248.489030,-368.855530,...,-150.828810,352.66696,69.07290,-160.042130,-67.788040,111.505005,252.334350,1753.999500,158.474030,-450.08978


In [10]:
import numpy as np
from sklearn.decomposition import NMF

# Generate random non-negative data
data = gene.to_numpy()

# Initialize the NMF model
n_components = 100
model_nmf = NMF(n_components=n_components, init='random', random_state=0)

# Fit the model to the data
W = model_nmf.fit_transform(data)
H = model_nmf.components_

# Display the results
print("Basis matrix (W):\n", W)
print("Coefficients matrix (H):\n", H)


c:\Users\phill\anaconda3\envs\tensorflow\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Basis matrix (W):
 [[2.6972830e-01 2.8575301e-01 3.2349173e-02 ... 1.7849728e-01
  7.2205931e-02 6.3250010e-04]
 [9.4617987e-01 1.7102477e-01 1.6105120e-01 ... 0.0000000e+00
  9.3340710e-02 0.0000000e+00]
 [4.3408936e-01 0.0000000e+00 2.9220533e-01 ... 1.0278109e-02
  4.1093561e-03 0.0000000e+00]
 ...
 [6.3596576e-01 0.0000000e+00 2.9008484e-01 ... 4.7547575e-03
  9.8687559e-02 0.0000000e+00]
 [1.0864531e+00 1.9762394e-01 3.7065756e-01 ... 5.7310890e-03
  2.9213486e-02 0.0000000e+00]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 0.0000000e+00]]
Coefficients matrix (H):
 [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.10467249 0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.04535175 0.         ... 0.         0.         0.00181273]
 [0.         0.      

In [11]:
W = pd.DataFrame(W)
W

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.269728,0.285753,0.032349,0.552654,0.124682,0.019776,0.001904,0.015233,0.165916,0.148393,...,0.000000,0.060630,0.216403,0.021437,0.029728,0.004525,0.020293,0.178497,0.072206,0.000633
1,0.946180,0.171025,0.161051,0.427533,0.243432,0.000000,0.000000,0.007518,0.240688,0.050733,...,0.000596,0.000000,0.067389,0.000000,0.027200,0.010487,0.015699,0.000000,0.093341,0.000000
2,0.434089,0.000000,0.292205,0.623748,0.211550,0.000000,0.000000,0.011240,0.000000,0.128447,...,0.029316,0.035785,0.003594,0.000000,0.000000,0.000070,0.012215,0.010278,0.004109,0.000000
3,0.729335,0.221870,0.000000,0.000000,0.257854,0.019506,0.045102,0.024270,0.000000,0.088816,...,0.043190,0.048438,0.000000,0.016525,0.006915,0.011320,0.012279,0.001098,0.034228,0.000888
4,0.656248,0.668048,0.471090,0.540120,0.022798,0.028723,0.088986,0.011827,0.007517,0.031618,...,0.048764,0.120444,0.298696,0.000000,0.012752,0.013183,0.025803,0.129973,0.047137,0.008924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35510,0.043546,0.184586,0.000000,0.000000,0.100512,0.005914,0.000000,0.006933,0.330994,0.000000,...,0.029424,0.035614,0.050792,0.000000,0.017568,0.012702,0.007590,0.000000,0.029065,0.001013
35511,0.514584,0.000000,0.167434,0.000000,0.140392,0.000000,0.035951,0.004200,0.150183,0.131524,...,0.028503,0.025795,0.000000,0.000000,0.005178,0.000000,0.022052,0.004517,0.034213,0.000718
35512,0.635966,0.000000,0.290085,0.035056,0.108897,0.000000,0.045690,0.000000,0.000000,0.071139,...,0.029825,0.019364,0.000000,0.025793,0.003166,0.004230,0.000132,0.004755,0.098688,0.000000
35513,1.086453,0.197624,0.370658,0.077014,0.221128,0.002844,0.022769,0.006905,0.114654,0.129683,...,0.000000,0.000000,0.054695,0.000000,0.003671,0.004837,0.012339,0.005731,0.029213,0.000000


In [12]:

# es_callback = EarlyStopping(monitor= 'val_auc', patience=20, restore_best_weights=True)
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=50, monitor='loss', min_delta=100)

history = model.fit(W,AE_tcr, 
                epochs=1400, 
                batch_size=128, 
                shuffle = True,
                # callbacks=[es_callback, checkpoint,reduce_learning_rate])
                # callbacks=[reduce_learning_rate]
                )

Epoch 1/1400
278/278 [==============================] - 7s 12ms/step - loss: 333368.5000
Epoch 2/1400
278/278 [==============================] - 3s 11ms/step - loss: 331840.7500
Epoch 3/1400
278/278 [==============================] - 3s 11ms/step - loss: 306339.4688
Epoch 4/1400
278/278 [==============================] - 3s 11ms/step - loss: 234333.7500
Epoch 5/1400
278/278 [==============================] - 3s 11ms/step - loss: 233093.3594
Epoch 6/1400
278/278 [==============================] - 3s 11ms/step - loss: 232244.7969
Epoch 7/1400
278/278 [==============================] - 3s 12ms/step - loss: 231672.8438
Epoch 8/1400
278/278 [==============================] - 3s 12ms/step - loss: 230987.9688
Epoch 9/1400
278/278 [==============================] - 3s 12ms/step - loss: 230183.6875
Epoch 10/1400
278/278 [==============================] - 3s 12ms/step - loss: 229105.6406
Epoch 11/1400
278/278 [==============================] - 3s 12ms/step - loss: 227607.9688
Epoch 12/1400
278/2

In [13]:
for layer in model.layers:
    print(layer.name)

input_1
dense
reshape
conv2d
conv2d_1
flatten
dense_1
reshape_1
conv2d_transpose
conv2d_transpose_1
dropout
flatten_1
dense_2
dense_3
dense_4


In [23]:
from tensorflow.keras.models import Model
latent_model = Model(inputs=input_gex, outputs=hidden_layer)


In [15]:
model.predict( W.iloc[1:2])

1/1 [==============================] - 0s 284ms/step


array([[1.31264984e+02, 0.00000000e+00, 1.91385635e+02, 0.00000000e+00,
        0.00000000e+00, 3.98622528e+02, 3.70863464e+02, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 5.12155151e+02, 4.74552277e+02, 1.77394989e+02,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 1.25315186e+02, 5.57568321e+01,
        0.00000000e+00, 0.00000000e+00, 3.38392944e+02, 0.00000000e+00,
        0.00000000e+00, 7.22407349e+02, 0.00000000e+00, 1.46514270e+03,
        0.00000000e+00, 2.23223495e+02, 1.43386917e+01, 0.00000000e+00,
        1.40667587e+02, 5.35112976e+02, 4.92976654e+02, 5.39690056e+01,
        6.28855713e+02, 0.00000000e+00, 0.00000000e+00, 6.71266235e+02,
        7.17289062e+02, 1.21579475e+02, 2.59756107e+01, 0.00000000e+00,
        0.00000000e+00, 4.45819778e+01, 0.00000000e+00, 0.00000000e+00,
        4.36872284e+02, 3.59081207e+02, 0.00000000e+00, 5.466134

In [16]:
model.predict( W.iloc[4:5])

1/1 [==============================] - 0s 19ms/step


array([[   0.      ,  241.37297 ,    0.      ,    0.      ,  302.0685  ,
           0.      ,  131.81558 ,    0.      ,    0.      ,    0.      ,
           0.      ,    0.      ,    0.      ,   88.75305 ,    0.      ,
           0.      ,    0.      ,    0.      ,    0.      ,  607.5684  ,
           0.      ,    0.      ,    0.      ,    0.      ,    0.      ,
           0.      ,  424.03433 ,    0.      ,    0.      ,    0.      ,
           0.      ,    0.      ,    0.      ,  357.60678 ,  431.24225 ,
         244.07965 ,   17.801702,    0.      ,    0.      ,    0.      ,
         870.7848  ,    0.      ,  231.49942 ,    0.      ,    0.      ,
           0.      ,  575.5794  ,    0.      ,  126.14164 ,  575.6279  ,
          14.614671,   50.178303,  468.7765  ,  170.71234 ,    0.      ,
           0.      ,    0.      ,  182.7847  ,   33.410492,    0.      ,
           0.      ,    0.      ,    0.      ,    0.      ,    0.      ,
           0.      ,    0.      , 2931.9575  ,    0

In [17]:
integration_pred = latent_model.predict( W)

1110/1110 [==============================] - 2s 1ms/step


In [24]:
pd.DataFrame(integration_pred[1:50,1:50])

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,0.0,0.000000,0.0,35.472549,0.0,0.000000,0.0,0.0,0.0,0.798549,...,0.0,105.749825,19.453865,0.0,32.836594,0.0,15.519721,0.000000,0.0,0.0
1,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.0,64.104187,2.041360,0.0,62.545567,0.0,50.675880,0.000000,0.0,0.0
2,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,23.370670,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
3,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,37.314919,...,0.0,43.495281,71.366272,0.0,0.000000,0.0,75.383720,0.000000,0.0,0.0
4,0.0,0.000000,0.0,6.303603,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.0,0.000000,72.740868,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
5,0.0,0.000000,0.0,0.000000,0.0,23.242109,0.0,0.0,0.0,5.023887,...,0.0,11.910394,33.495125,0.0,45.537582,0.0,10.590414,0.000000,0.0,0.0
6,0.0,0.000000,0.0,36.465187,0.0,0.000000,0.0,0.0,0.0,7.256657,...,0.0,41.674850,47.641575,0.0,4.274441,0.0,20.562767,0.000000,0.0,0.0
7,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,...,0.0,47.325619,0.000000,0.0,94.540611,0.0,78.171875,76.400444,0.0,0.0
8,0.0,5.691920,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,49.877533,...,0.0,85.156822,114.295082,0.0,86.029755,0.0,18.975483,0.000000,0.0,0.0
9,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,71.683403,...,0.0,9.890483,65.132507,0.0,64.267532,0.0,39.269470,3.980707,0.0,0.0


In [19]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])

In [20]:
integration_pred.shape

(35515, 225)

In [21]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])
integration_pred.shape

(35515, 225)

In [22]:
pd.DataFrame(integration_pred)

,0,1,2,3,4,5,6,7,8,9,...,215,216,217,218,219,220,221,222,223,224
0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,102.337921,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
1,67.945328,0.0,0.000000,0.0,35.472549,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,19.417398,0.000000,0.0
2,25.923313,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
3,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,61.579628,0.0,0.0,0.0,0.0,0.0,0.0,48.461872,0.000000,0.0
4,42.633919,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35510,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,50.004463,0.0,0.0,0.0,0.0,0.0,0.0,48.376793,0.000000,0.0
35511,95.926804,0.0,0.000000,0.0,6.670987,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
35512,15.759350,0.0,0.000000,0.0,24.743168,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
35513,77.846939,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,20.496489,0.0


In [25]:
pd.DataFrame(integration_pred).to_csv("integration_pred_new_method_gex_to_TCR_beta_chain_10X_donor_1_only.csv", index=False)